# CodeSwitch-TTS: A Hybrid Non-Autoregressive Draft and Autoregressive Verification Framework for Code-Mixed Tamil-English Speech Synthesis

This notebook runs the unified **CodeSwitch-TTS** speculative decoding pipeline designed for Tanglish (Tamil-English code-mixed speech) that achieves **<300ms Time-To-First-Audio (TTFA)** on NVIDIA T4 GPUs.

### Instructions:
1. **Accelerator**: Select **GPU T4 x2** (Notebook Settings -> Accelerator -> GPU T4).
2. **Hugging Face Token** (Optional for faster downloads): In the top menu, go to **Add-ons -> Secrets**, add a secret with label  and your Hugging Face token.
3. **Run All**: Click **Run All** to install staged dependencies, load the hybrid pipeline, synthesize audio, and benchmark latency.


## 1. Hardware & System Diagnostics

In [ ]:
import subprocess, sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Running on CPU mode. Please enable GPU T4 in Kaggle notebook settings for full acceleration.")


## 2. Hugging Face Authentication (via Kaggle Secrets)

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("✓ Hugging Face login successful via Kaggle Secrets!")
    else:
        print("ℹ HF_TOKEN not set in Kaggle Secrets; proceeding with anonymous downloads.")
except Exception as e:
    print(f"ℹ Proceeding with public access ({e}).")


## 3. Clone Repository & Staged Dependency Installation

In [ ]:
import os, sys, importlib.metadata as md

REPO_URL = "https://github.com/tripathiji1312/codeSwitch.git"
CLONE_DIR = "/kaggle/working/codeSwitch"

# --- 1. Always sync the repository (fast-forward the clone, or clone fresh) ---
if os.path.exists(os.path.join(CLONE_DIR, ".git")):
    if os.system(f"git -C {CLONE_DIR} fetch origin main") != 0:
        os.system(f"rm -rf {CLONE_DIR}")
        os.system(f"git clone {REPO_URL} {CLONE_DIR}")
    else:
        os.system(f"git -C {CLONE_DIR} reset --hard FETCH_HEAD")
else:
    os.system(f"rm -rf {CLONE_DIR}")
    os.system(f"git clone {REPO_URL} {CLONE_DIR}")

# --- 2. Decide from the installed state (not a marker file) whether to (re)install ---
def _deps_ok() -> bool:
    try:
        if tuple(map(int, md.version("protobuf").split(".")[:2])) < (5, 26):
            return False                        # no google.protobuf.runtime_version
        if md.version("transformers") != "4.46.1":
            return False
        md.version("parler-tts")                # raises if parler-tts is missing
        md.version("sentencepiece")
        md.version("jiwer")                     # evaluation suite (CER/WER)
        md.version("matplotlib")                # paper figures
        return True
    except Exception:
        return False

if not _deps_ok():
    print("Installing dependencies (staged to avoid package conflicts)...")
    # Stage 1: parler-tts from GitHub (brings its transformers/audiotools chain)
    os.system("pip install -q -U git+https://github.com/huggingface/parler-tts.git")
    # Stage 2: pin transformers to the exact version parler-tts v0.2.2 targets
    os.system("pip install -q 'transformers==4.46.1'")
    # Stage 3: remaining audio/scientific libs
    os.system("pip install -q descript-audio-codec soundfile scipy accelerate sentencepiece")
    # Stage 3b: evaluation-suite extras (CER/WER + IEEE-style figures)
    os.system("pip install -q jiwer matplotlib seaborn pandas")
    # Stage 4: protobuf LAST and >= 5.26 (it ships google.protobuf.runtime_version).
    # parler-tts -> descript-audiotools pins protobuf<5 and silently downgrades it to
    # 4.25.9, which broke the Parler-TTS import and forced the noise-only fallback.
    os.system("pip install -q 'protobuf==5.29.5'")
    print("✓ Dependencies installed!")
    print("⚠ The kernel now restarts so the new protobuf/transformers take effect.")
    print("👉 Click 'Run All' again — installation is skipped on the second pass.")
    os._exit(0)  # hard restart; a stale protobuf 4.x cannot be swapped in-process
else:
    print("✓ Dependencies already installed.")

print(f"   protobuf:     {md.version('protobuf')}  (needs >= 5.26)")
print(f"   transformers: {md.version('transformers')}")

# --- 3. Sanity-check the Parler-TTS import (the real cause of the static-noise runs) ---
try:
    from parler_tts import ParlerTTSForConditionalGeneration  # noqa: F401
    print("✓ parler_tts import OK")
except Exception as e:
    print(f"⚠ parler_tts import FAILED: {type(e).__name__}: {e}")
    print("  Fix: !pip install -q 'protobuf==5.29.5'  then restart the session (Stop & Run All).")

# --- 4. Make the packaged code importable ---
for path_candidate in [CLONE_DIR, ".", ".."]:
    if os.path.exists(os.path.join(path_candidate, "codeswitch_tts")):
        if path_candidate not in sys.path:
            sys.path.insert(0, path_candidate)
        print(f"✓ Root path configured: {path_candidate}")
        break


## 4. Initialize CodeSwitch-TTS Hybrid Pipeline

In [ ]:
import torch
from codeswitch_tts.config import CodeSwitchTTSConfig
from codeswitch_tts.engine.streaming_pipeline import ChunkedStreamingPipeline
from codeswitch_tts.utils.audio import save_wav
from IPython.display import Audio, display

print("Initializing CodeSwitch-TTS...")
config = CodeSwitchTTSConfig()
pipeline = ChunkedStreamingPipeline(config=config)

if getattr(pipeline, "speech_available", False):
    print("✓ Pipeline ready! Authentic Indic Parler-TTS speech backend is active.")
else:
    print("⚠ Indic Parler-TTS did not load: synthesis would only produce random-weight")
    print("  noise. Apply the fix printed above, restart the session, and re-run.")


## 5. Streaming Synthesis on 3 Research Test Utterances with Audio Playback

In [ ]:
test_utterances = [
    {
        "category": "Native Tamil",
        "text": "உங்கள் பயணம் வெற்றிகரமாக முடிந்தது.",
        "filename": "sample1_native_tamil.wav",
    },
    {
        "category": "Romanized Tanglish",
        "text": "Unga cab ippo pickup location-ku vandhuchu.",
        "filename": "sample2_romanized_tanglish.wav",
    },
    {
        "category": "Transit Code-Mixed",
        "text": "Driver-ku OTP 5291 sollunga, total fare Rs. 245.",
        "filename": "sample3_transit_codemixed.wav",
    },
]

os.makedirs("output_audio", exist_ok=True)
benchmark_results = []

# ---------------------------------------------------------------------------
# Warm-up pass.  The first synthesis after loading the model pays one-off CUDA
# kernel / allocator / description-cache costs (on a T4: ~4.3 s vs ~0.7 s once
# warm - which previously made the first benchmarked category look ~20x slower
# than the others).  Two throwaway utterances - including a long code-mixed one
# so the DAC decoder is exercised at full length - are synthesised with timings
# discarded, so the benchmark below reports steady-state numbers.
# ---------------------------------------------------------------------------
print("Warming up (CUDA kernels, allocator, DAC, description cache)...")
for warm_text in (
    "இது ஒரு வெப்பநிலை சோதனை, முடிவுகளில் சேர்க்கத் தேவையில்லை.",
    "Warm-up only: driver-ku OTP 1234 sollunga, total fare Rs. 500, vehicle TN 09 AB 1234.",
):
    try:
        pipeline.synthesize(warm_text)
    except Exception as exc:
        print(f"  [warmup] skipped: {type(exc).__name__}: {exc}")
print("Warm-up done (timings discarded).")
print("=" * 75)


for i, sample in enumerate(test_utterances, 1):
    print("=" * 75)
    print(f" [{i}/3] Category: {sample['category']}")
    print(f" Raw Input:      \"{sample['text']}\"")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    result = pipeline.synthesize(sample["text"])
    benchmark_results.append((sample, result))
    
    print(f" Normalized:     \"{result.normalized_text}\"")
    print(f" Detected Reg:   {result.detected_language}")
    print(f" TTFA (ms):      {result.ttfa_ms:.1f} ms  (Target: <300 ms)")
    print(f" Total Latency:  {result.total_latency_ms:.1f} ms")
    print(f" Audio Duration: {result.duration_sec:.2f} s")
    print(f" RTF:            {result.real_time_factor:.3f}")
    print(f" CB0 Acceptance: {result.cb0_acceptance_rate:.1f}%")
    
    # Save Audio
    out_path = os.path.join("output_audio", sample["filename"])
    save_wav(out_path, result.audio_waveform, sample_rate=result.sample_rate)
    print(f" ✓ Saved to: {out_path}")
    
    # Display inline interactive audio player
    print(f" Listen ({sample['category']}):")
    display(Audio(result.audio_waveform, rate=result.sample_rate))


## 6. Latency Analysis & Architectural Comparison Table

In [ ]:
print("=" * 80)
print(" BENCHMARK SUMMARY & LATENCY ANALYSIS")
print("=" * 80)
header = f"{'Category':<22} | {'TTFA (ms)':<10} | {'Total (ms)':<11} | {'Audio (s)':<10} | {'RTF':<8} | {'CB0 Accept %'}"
print(header)
print("-" * 80)
for sample, res in benchmark_results:
    print(f"{sample['category']:<22} | {res.ttfa_ms:<10.1f} | {res.total_latency_ms:<11.1f} | {res.duration_sec:<10.2f} | {res.real_time_factor:<8.3f} | {res.cb0_acceptance_rate:.1f}%")
print("-" * 80)

ttfas = [res.ttfa_ms for _, res in benchmark_results]
if len(ttfas) >= 2:
    warm = ttfas[1:]
    print(f" TTFA breakdown: first measured utterance = {ttfas[0]:.1f} ms | "
          f"warm mean = {sum(warm) / len(warm):.1f} ms | warm min = {min(warm):.1f} ms")
    print(" (cold-start overhead was absorbed by the warm-up cell above)")
    print("-" * 80)

print()
print("For the paper: run the full suite (cells below) - 60 utterances x 3 configurations,")
print("true (teacher-forced) + synthetic CB0 acceptance per linguistic zone, IEEE figures")
print("and LaTeX tables written to paper_figures/.")
print()
print("ARCHITECTURAL LATENCY COMPARISON - DESIGN TARGETS (not measured here):")
print(" ┌───────────────────────────────────────┬──────────────────┬──────────────────┐")
print(" │ Metric                                │ AR Baseline      │ CodeSwitch-TTS   │")
print(" ├───────────────────────────────────────┼──────────────────┼──────────────────┤")
print(" │ First Chunk TTFA                      │ ~3,200 ms        │ ~230 - 260 ms    │")
print(" │ Speculative Decoding Codebook Target  │ All 9 CBs (Fails)│ Codebook 0 Only  │")
print(" │ Codebook Rejection Collapse Risk      │ High (>77% rej.) │ Target: low rej. │")
print(" │ Residual Codebook 1-8 Generation      │ Sequential AR    │ Parallel NAR     │")
print(" │ Real-Time Factor (RTF)                │ ~1.20 (Lagging)  │ ~0.24 (target)   │")
print(" │ TTFA Latency Reduction Target         │ Baseline (1.0x)  │ >92% (13.6x fast)│")
print(" └───────────────────────────────────────┴──────────────────┴──────────────────┘")
print(" ⚠ The box above lists plan targets; quote the measured TTFA / CB0 acceptance from")
print("   the table above and from paper_figures/tables.tex, never the targets as results.")


## 7. Interactive Custom Tanglish Synthesis

In [ ]:
# Test any custom Tanglish or transit text here:
custom_text = "Driver anna, pickup location reach aana udane call pannunga, OTP 4821."

print(f'Input Text: "{custom_text}"')
custom_res = pipeline.synthesize(custom_text)
print(f'Normalized: "{custom_res.normalized_text}"')
print(f"TTFA:       {custom_res.ttfa_ms:.1f} ms")
print(f"Acceptance: {custom_res.cb0_acceptance_rate:.1f}%")
display(Audio(custom_res.audio_waveform, rate=custom_res.sample_rate))


## 8. Self-Distillation of the Draft / Residual Models

Collects teacher DAC tokens for a bootstrap Tanglish corpus (cached per sentence in
`teacher_cache/`, safe to re-run) and trains both NAR modules:

* `codeswitch_ckpt/draft_cb0.pt` — Codebook-0 draft transformer (~17M params)
* `codeswitch_ckpt/residual_cb1_8.pt` — residual decoder for Codebooks 1–8 (~7M params)
* `codeswitch_ckpt/meta.json` — fitted text-length model + training stats

Expect **35–55 min** on a T4. The loop prints an ETA every 10 sentences; Ctrl-C is safe.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
!python train_draft_distill.py --num-sentences 300 --ckpt-dir codeswitch_ckpt


## 9. Full Benchmark Evaluation (60 utterances × 3 configs)

Runs Config **A** (pure AR, real `ParlerTTSStreamer`), **B** (pure NAR) and
**C** (CodeSwitch-TTS hybrid speculative) on the imported 60-utterance Tanglish
benchmark, then writes IEEE-style figures and LaTeX tables into `paper_figures/`.

Results are appended atomically after every utterance — just re-run the cell to resume.
Expect **25–40 min** on a T4 (whisper-small ASR included).


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
!python run_full_evaluation.py --ckpt codeswitch_ckpt


## 10. Collect Artifacts

Bundles the figures, tables, metrics and audio samples for download.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# Figures + tables + metrics + audio samples (checkpoint weights excluded)
!zip -qr /kaggle/working/codeswitch_artifacts.zip paper_figures eval_results eval_audio codeswitch_ckpt -x 'codeswitch_ckpt/*.pt'
!ls -lh /kaggle/working/codeswitch_artifacts.zip
